In [1]:
%run ../../Latex_macros.ipynb

<IPython.core.display.Latex object>

# RLLM-9: Sample, filter, imitate

## RL for LLMs, Part 2 (RFT alternative methods), session 9 of 14

## What this session teaches

A model can improve by training on its own completions, as long as a reward decides which completions count.

The recipe has three steps. The model writes several completions to each prompt. A reward scores each one. A **filter** is a rule that gives each completion a weight of one or zero, and keeps the completions with weight one.

Then ordinary supervised fine-tuning trains on the kept completions, as if they were examples to copy. The filter gives a dropped completion weight zero, which drops it: it adds no push, up or down.

Filters differ in what they keep: every completion that reaches one fixed reward, called the **threshold**, or only the best completion for each prompt.

Equal weights on a model's own completions push nowhere on average, and the filter breaks that. For a fixed threshold, one training step moves the model, on average, the same way a policy-gradient step does.

The catch is that a dropped completion never teaches, however instructive its failure.

The question to carry away: can a worse completion teach, if we set it against a better one?

By the end you can

- say what a filter is, and what weight zero does to a completion
- say in what sense, and for which filter, this loop is a policy-gradient step
- say what each of the two filters keeps and what it throws away

**Where this session fits: the topics of the series, in order**

<img src="images/rllm9v3/series_strip_8.png" width=88% alt="Strip of the topics of the series with topic 8, self-training, highlighted.">

- this session is topic 8: the model trains on its own completions that a reward approves


## Setting and words

**Why this slide?** The loop uses terms that the opening does not define; each is said by what it does.

| term | what it means here |
|:--|:--|
| **round** | one pass of write, filter and train; the next round writes fresh completions from the updated model |
| **kept set** | the filtered completions of one round: the only data the training step sees |
| **G** | the number of completions the model writes for one prompt |
| **best of G** | the highest-reward completion among the G written for one prompt (a tie: the first) |
| **temperature 1** | writing completions with the model's own probabilities, unchanged (Recall from Intro C1) |
| **self-training** | the whole family of loops: the model trains on its own completions that a reward approves |
| **threshold filter** | this session's name for the filter with one fixed threshold; papers call the loop with it self-distillation (a word with other meanings, see the skipped aside on names) |
| **best-of-G filter** | keeps the best of the G completions of each prompt; the paper calls it RAFT |

## How we will teach it: the map

**Why this slide?** One loop, whose only free part is the filter. Four steps, in order; the filter comes in two cases that later steps compare.

<img src="images/rllm9v3/rllm9v3_map.png" width=66% alt="The loop write, reward, filter, SFT drawn once, with the filter split into a threshold filter and a best-of-G filter, and under it four sections: Loop, Two filters, Why it moves, What a filter drops.">

- **Loop:** the three steps, and what weight one or zero does
- **Two filters:** the **threshold filter** and the **best-of-G filter**
- **Why it moves:** the filter breaks RLLM-8's zero, and in what sense the loop is a policy-gradient step
- **What a filter drops:** what weight zero throws away

## Loop: sample, reward, filter, imitate

*You are here: Loop.*

<img src="images/rllm9v3/rllm9v3_bar_1.png" width=55% alt="Section bar with Loop highlighted.">

**Why this slide?** A reward can improve a model without any RL machinery.

<img src="images/rllm9v3/rllm9v3_loop.png" width=62% alt="Prompt, the model writes G completions, a reward scores each, a filter keeps some with weight 1 and drops the rest with weight 0, SFT trains on the kept, then the updated model writes the next round.">

- the model writes G completions for each prompt, and a reward scores each one
- the filter gives each completion weight 1 (kept) or weight 0 (dropped)
- SFT (RLLM-1) trains on the kept completions, as examples to copy
- the parameters change only in the SFT step: the reward enters through the filter

## Loop: weight one or zero, then ordinary SFT

**Why this slide?** It says what this hybrid gives: a reward picks the data, and plain SFT learns from it.

<img src="images/rllm9v3/rllm9v3_kept.png" width=60% alt="Four completions of one prompt with rewards 0.9, 0.8, 0.4, 0.2 and weights 1, 1, 0, 0; the two kept completions go into an SFT batch.">

- a kept completion gets weight 1 on every token
- a dropped one adds no push, up or down
  - it is lowered only indirectly, by normalisation (the probabilities sum to 1)
- there is no critic, ratio, clip or baseline to train (our interpretation)
- a verifier's pass-or-fail is enough: weight 1 for a pass, weight 0 for a fail

## Two filters: the threshold filter (self-distillation)

*You are here: Two filters.*

<img src="images/rllm9v3/rllm9v3_bar_2.png" width=55% alt="Section bar with Two filters highlighted.">

**Why this slide?** The first case: keep whatever reaches one fixed threshold.

```python
policy = sft_checkpoint
for rnd in range(N):
    kept = []
    for prompt in prompts:                     # every training prompt
        completions = sample(policy, prompt, G, temperature=1.0)   # G per prompt
        for c in completions:
            if reward(c) >= threshold:         # one fixed value, the same for every prompt
                kept.append((prompt, c))
    policy = sft(policy, kept)                 # imitate the kept: several steps on this one batch
```

- **easy prompts** give many kept completions, **hard prompts** few or none
- too high a threshold keeps nothing
- too low keeps everything, and we are back to equal weights with no push

## Two filters: the best-of-G filter (RAFT)

**Why this slide?** The second case, and the one line that differs.

```python
policy = sft_checkpoint
for rnd in range(N):
    kept = []
    for prompt in prompts:
        completions = sample(policy, prompt, G, temperature=1.0)
        best = max(completions, key=reward)    # best of G per prompt (a tie: the first)
        kept.append((prompt, best))            # every prompt contributes, always
    policy = sft(policy, kept)
```

- the outer loop is identical
- only the filter inside it differs
- the **threshold filter** keeps a prompt's completions only if something reaches the threshold
- the **best-of-G filter** ranks within the prompt: every prompt gives exactly one
- RAFT is reward-ranked fine-tuning (Dong et al., 2023)

## Two filters: the same twelve completions under each

**Why this slide?** Twelve completions, three prompts, four each: which ones does each filter keep?

<img src="images/rllm9v3/rllm9v3_filters.png" width=52% alt="Rewards of twelve completions for three prompts, then the weights the threshold filter gives (only prompt A) and the weights the best-of-G filter gives (one per prompt, including prompt C's 0.41).">

- with the threshold at 0.75, only prompt A's completions are kept
- the best-of-G filter keeps one completion per prompt, even prompt C's poor 0.41

## Why it moves: the filter breaks the zero

*You are here: Why it moves.*

<img src="images/rllm9v3/rllm9v3_bar_3.png" width=50% alt="Section bar with Why it moves highlighted.">

**Why this slide?** It links to RLLM-8: a reward must set the weights.

<img src="images/rllm9v3/filter_expected_step.png" width=50% alt="Toy policy: with equal weights the exact average push is zero; with a 0/1 filter weight the exact average push raises the actions that clear the threshold.">

<!-- seed-pay: zero-push-equal-weights -->
- from RLLM-8: equal weights on own completions push nowhere on average (left)
- a weight chosen by a reward gives a push (right)
  - 1 if the completion reaches the threshold, 0 if not
  - the kept completions are raised, the rest are left alone

## Why it moves: one policy-gradient step, for the threshold filter

**Why this slide?** This is the significant result, and where it stops.

> **For the threshold filter, one SFT step on the kept set is, in expectation, one policy-gradient step on "the probability that a completion reaches the threshold".**

- **why it matters:** plain supervised training on the kept completions climbs the same objective as RL with a 0-or-1 reward
  - so a method with no critic, no ratio and no clip still follows the policy-gradient direction (Recall RLLM-5)
- **the limits**
  - one expected step, and it improves only that probability
  - the threshold filter only: it is not derived for the best-of-G filter (a rank within the prompt)
  - no KL penalty holds the model near the start
  - several SFT steps on one batch go stale: RLLM-7, "How stale is too stale?"
- derivation and conditions: the skipped asides

## What a filter drops: no push down

*You are here: What a filter drops.*

<img src="images/rllm9v3/rllm9v3_bar_4.png" width=50% alt="Section bar with What a filter drops highlighted.">

**Why this slide?** It is the cost of the method.

<img src="images/rllm9v3/rllm9v3_drop.png" width=50% alt="The same four completions: filter weights 1, 1, 0, 0 and GRPO's signed weights positive, positive, negative, negative.">

- GRPO (RLLM-6) gives signed weights from the group mean: positive above it, negative below
- a filter has no explicit push down on a bad completion
- it has no magnitude: 0.81 and 0.95 both reach a 0.75 threshold
- it has no baseline (RLLM-5), so it cannot say what to expect
  - that is why dropped means weight zero, not negative: a negative weight says "worse than expected"
- what each filter's cost is, in practice, is our interpretation: it is not measured here

## What a filter drops: a failure never teaches

**Why this slide?** It sets up the question to carry away.

> **Keep this: both filters give a dropped completion weight zero, so a loser teaches nothing. Yet the difference between a winner and a loser is the richest signal there is.**
<!-- seed-plant: weight-zero-rejected -->

- a filter sees only what reaches the threshold, never why another completion failed
- set a loser against a winner on the same prompt, and the difference between them is visible
  - learning from that difference is **contrastive learning**, the subject of RLLM-10
- a **preference pair**, a better and a worse completion of one prompt, keeps both

## Placing the loop on RLLM-8's grid

**Why this slide?** The grid grows by one box each session.

<img src="images/rllm9v3/rllm9v3_grid.png" width=50% alt="RLLM-8's grid with this session's hybrid box spanning both columns of the own-samples row; RLLM-10 and RLLM-11 boxes dashed.">

- the hybrid box spans both columns of the own-samples row
  - a reward picks the completions, and the kept tokens are the reference that SFT copies
- how fine: a weight of 1 or 0 per completion, applied to every token of a kept completion
- the other boxes stay as in RLLM-8
- the placements are our interpretation of how each method is built, not a result

## Think about

*The loop leaves open what a filter does when rewards are nearly all equal, and what repeated rounds do. State a prediction for each; these are questions, not claims.*

1. **Predict first.** Method Y: *sample $G$ completions per prompt, keep the top 25% by reward within each prompt, SFT on them.*
   - which of this session's two filters is it closer to?
   - place it by whose samples, where the weights come from, and how fine they are
2. **Predict first.** Raise the threshold from 0.75 to 0.99
   - what happens to the size of the kept set, and to the noise of the update?
3. Both filters put weight 0 on dropped completions
   - could you use them **and** keep an SFT-style loss?
   - what would a pair (a *better* and a *worse* completion of one prompt) let you say that one kept completion cannot?
4. Design a small experiment to test whether **negative weights matter in practice**
   - what would you have fixed, and what would you measure?
5. **Predict first.** A prompt where **every** completion scores 0: what does each filter keep?
   - what does that say about how each filter treats **hard prompts**?

## What we established, and what it sets up

- **Established**
  - a filter is a weight chosen by a reward, so it breaks the zero push
  - the threshold filter gives a policy-gradient step, in expectation, for one step
  - the best-of-G filter keeps one completion per prompt
- **Limit of this session**
  - dropped completions get weight zero, so only the kept ones teach
  - the equivalence covers one expected step
- **Sets up RLLM-10, Preference pairs and DPO:** can the dropped completions teach?
  - a preference pair keeps both a better and a worse completion

## Aside 1, justification: why the threshold filter is a policy-gradient step

*For your own review; not worked through in session. Our derivation from the standard identity; the conditions are in aside 2.*

1. Let $\kappa$ be the filter's threshold and $R(\y)=1$ if $\rew(\y)\ge\kappa$, else 0. The objective is $J(\theta)=\Exp{\y\sim\pi_\theta}[R(\y)]$: the probability that a sampled completion clears the threshold
2. The policy-gradient identity (RLLM-5): $\nabla J=\Exp{\y\sim\pi_\theta}[R(\y)\,\nabla_\theta\log\pi_\theta(\y)]$
3. The SFT loss on the kept set is $-\sum R(\y)\log\pi_\theta(\y)$ over $n$ samples
   - in expectation its gradient is $-n\,\nabla J$: **the same expression**
- The figure's toy checks it: the exact average push of the filter equals the numerical gradient of $J$ (asserted in the figure script)
- The reward is not in the loss; it enters the gradient only as the 0/1 multiplier $R$


## Aside 2, conditions on the equivalence

*For your own review; it qualifies aside 1.*

**Result, in words.** Descending the SFT loss on the kept set is, **in expectation**, a step of **policy gradient on a 0/1 reward with no baseline**, even though the reward enters *only* through the filter. It is *not* "filter-then-SFT is RL". It holds only under these conditions ($\kappa$ is the `threshold` in the pseudocode):

- **Expected direction, one step.** A round takes *many* SFT steps on one fixed batch
  - after the first step the batch is no longer on-policy, and nothing corrects for that (PPO's ratio and clipping, RLLM-5, exist for this)
  - the next round's *fresh* completions are on-policy again
- **A thresholded objective.** $J$ is the probability of clearing $\kappa$, not expected reward: 0.81 and 0.95 count the same, and everything below $\kappa$ is invisible
- **The threshold filter only.** RAFT's best-of-$G$ is a *rank*-based weight, depending on the other completions of its prompt
  - the identity above was not derived for it (our interpretation)
- **A baseline that is constant within a prompt** (not depending on the completion's own value) leaves the expected direction unchanged and changes the variance
  - GRPO's group mean is *approximately* such a baseline
  - its division by the standard deviation, and by completion length, are separate choices that *do* change who gets weight
- **Sum vs. mean.** Averaging the SFT loss over each prompt's kept completions, instead of summing, keeps each prompt's direction
  - but prompts are re-weighted (if the number kept is treated as fixed)
- **No baseline** is why every weight is $\ge 0$; GRPO's group mean is where its negative weights come from


## Aside 3, sources: expert iteration and STaR

*For your own review.*

- **Expert iteration** (Anthony et al., 2017): an "expert" (there, a tree search) improves on the current policy, and a fast "apprentice" imitates it; repeat
- **STaR** (Zelikman et al., 2022): the model writes step-by-step reasoning; keep only reasoning that ends in the correct answer; fine-tune on it; repeat
  - STaR also hints the answer to the model on failed prompts; we skip that
- Both are close to sample, filter, imitate. The details differ, for example whether a round continues from the current policy or restarts from the base model
  - a fixed correct-or-not reward plays the constant threshold (**our interpretation**)
- **RAFT** (Dong et al., 2023) is the best-of-$G$ variant in the session


## Aside 4, the same loop in the DeepSeekMath table

*For your own review; we will not work through it in session.*

- **Source**: Shao et al. (2024), the DeepSeekMath paper, section 5.2.1 ("Towards to a Unified Paradigm"), with a table of methods
- Their gradient, up to notation (and a $1/|o|$ **length normalization** that we leave out):
  $$\nabla_\theta J_{\mathcal A}(\theta)=\Exp{(q,o)\sim\mathcal D}\Big[\frac{1}{|o|}\sum_t GC_{\mathcal A}(q,o,t)\,\nabla_\theta\log\pi_\theta(o_t\mid q,o_{<t})\Big]$$
  - $GC$ is the **gradient coefficient**: our weight $w_\tt$
  - the **data source** $\mathcal D$ is a **separate component**: our whose-samples axis from RLLM-7
- They name **three components** that differ between methods: the data source, the algorithm (how $GC$ is computed), and the reward signal
- Their table, in our words:

| Method | Data source | Gradient coefficient $GC$ |
|---|---|---|
| SFT | human-written demonstrations | $1$ |
| RFT | completions from the **SFT model** | $1$ if the answer is correct, else $0$ |
| Online RFT | completions from the **current** model | the same $0/1$ |
| DPO | pairs of completions | $\sigma(-z)$-type weight: **up** on the winner's tokens, **down** on the loser's |
| PPO | completions from the current model | the advantage $\hat{\advseq}_\tt$ |
| GRPO | groups of completions from the current model | the group-relative advantage, plus a KL term |

- **"RFT" in this table is the paper's name for rejection-sampling fine-tuning**: sample, keep the correct answers, then SFT on them
  - it is **not** this series' "reinforcement fine-tuning" (RLLM-1)
- **Reading it**: RFT and Online RFT differ only in the **data** column
  - Online RFT is this session's loop with a correctness filter
  - SFT and RFT differ in the **coefficient**. That is our two axes
- Their paper uses this to **compare methods**, as we do. The details of each coefficient are in its appendix


## Aside 5, names that clash, and a trap in RAFT

*For your own review; not worked through in session.*

- **Self-distillation** here means the threshold filter's loop, with a threshold filter. It is **not** self-critique (RLLM-3), and papers also use the word for one model taught with extra context: that is RLLM-12's setting, not this loop
- **RFT**: DeepSeekMath's "RFT" means rejection-sampling fine-tuning (this loop with a correctness filter). This series' RFT is reinforcement fine-tuning (RLLM-1)
- **A trap in RAFT**: a group in which every completion scores 0 still keeps one completion (the first of the tie), so RAFT trains on a failure


## Reading for this session

- **Suggested**: for a paper, at least the **abstract and introduction**. For other items, the parts named
  - [Zelikman, Wu, Mu and Goodman (2022), *STaR: Bootstrapping Reasoning With Reasoning*](https://arxiv.org/abs/2203.14465): keep what works, imitate
  - [Dong et al. (2023), *RAFT: Reward rAnked FineTuning for Generative Foundation Model Alignment*](https://arxiv.org/abs/2304.06767): ranked completions as SFT data
- **Additional** (4 more, for digging deeper), and the other sources this session cites: `reading_list.ipynb`
- Links were checked in October 2026; tell us if one has moved
